In [2]:
!pip install ddgs trafilatura -q

In [27]:
import os
from openai import OpenAI
from dotenv import load_dotenv
from openai import OpenAI
import json
from pprint import pprint
from ddgs import DDGS
import trafilatura
from pprint import pprint
from IPython.display import Markdown, display



load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if OPENAI_API_KEY is None:
    raise Exception("API Key is missing!")
else:
    print("Key is: " + OPENAI_API_KEY[:8])

client = OpenAI(api_key=OPENAI_API_KEY)

Key is: sk-proj-


In [28]:

def search_web(query):
    ddgs = DDGS()

    results = ddgs.text(query, max_result = 5)
    pprint(f"Got results: {results}")
    return json.dumps(results, indent=2)

In [29]:
def fetch_url(url: str):

    downloaded = trafilatura.fetch_url(url)

    if downloaded:
        text = trafilatura.extract(downloaded)
        print(text)
        if text: 
            print(f" \u2705 Got Text")
            return text
    print(f"\u274C fail to fetch text from URL {url}. ")
    return (f"Could not extract text from url {url}. Try different source.")


In [30]:
search_web("AI in healthcare in 2030")

("Got results: [{'title': 'The Growing Role of Artificial Intelligence in "
 "Healthcare', 'href': "
 "'https://mayomagazine.mayoclinic.org/2026/04/ai-in-healthcare/', 'body': "
 '"Artificial intelligence (AI) is a technological revolution that promises to '
 'redefine the contours of modern medicine. From accelerating diagnoses to '
 "personalizing treatment plans, AI is poised to tackle some of healthcare's "
 'most pressing challenges. The Rise of AI in Healthcare Alan Turing first '
 'coined the term “artificial intelligence” in the 1950s. Two decades later, '
 'AI took its initial ..."}, {\'title\': \'Adoption of artificial intelligence '
 "in healthcare: survey of ...', 'href': "
 "'https://pmc.ncbi.nlm.nih.gov/articles/PMC12202002/', 'body': 'The US "
 'healthcare system faces significant challenges, including clinician burnout, '
 'operational inefficiencies, and concerns about patient safety. Artificial '
 'intelligence (AI), particularly generative AI, has the potential to ad

'[\n  {\n    "title": "The Growing Role of Artificial Intelligence in Healthcare",\n    "href": "https://mayomagazine.mayoclinic.org/2026/04/ai-in-healthcare/",\n    "body": "Artificial intelligence (AI) is a technological revolution that promises to redefine the contours of modern medicine. From accelerating diagnoses to personalizing treatment plans, AI is poised to tackle some of healthcare\'s most pressing challenges. The Rise of AI in Healthcare Alan Turing first coined the term \\u201cartificial intelligence\\u201d in the 1950s. Two decades later, AI took its initial ..."\n  },\n  {\n    "title": "Adoption of artificial intelligence in healthcare: survey of ...",\n    "href": "https://pmc.ncbi.nlm.nih.gov/articles/PMC12202002/",\n    "body": "The US healthcare system faces significant challenges, including clinician burnout, operational inefficiencies, and concerns about patient safety. Artificial intelligence (AI), particularly generative AI, has the potential to address these .

In [31]:
fetch_url("https://www.linkedin.com/pulse/ai-healthcare-2030-opportunities-challenges-strategic-riken-shah-pk86f")

AI in Healthcare 2030: Opportunities, Challenges, and Strategic Insights
Healthcare is entering a transformative era. The volume of patient data, medical literature, and imaging is growing exponentially, creating new pressures for clinicians and administrative staff. Workforce shortages and increasing patient demand are only amplifying these challenges.
We explored these issues in a recent webinar featuring Jan Beger , a healthcare AI veteran with over 20 years of experience. We discussed how AI can help health systems manage these pressures, streamline workflows, and improve patient outcomes.
In practice, we’ve helped hospitals implement AI tools that prioritize critical imaging, optimize scheduling, and identify high-risk patients for follow-up. These solutions reduce administrative burdens, enhance operational efficiency, and allow clinicians to focus on delivering quality care.
At OSP, we focus on integrating AI seamlessly into workflows, ensuring technology enhances human capabili

"AI in Healthcare 2030: Opportunities, Challenges, and Strategic Insights\nHealthcare is entering a transformative era. The volume of patient data, medical literature, and imaging is growing exponentially, creating new pressures for clinicians and administrative staff. Workforce shortages and increasing patient demand are only amplifying these challenges.\nWe explored these issues in a recent webinar featuring Jan Beger , a healthcare AI veteran with over 20 years of experience. We discussed how AI can help health systems manage these pressures, streamline workflows, and improve patient outcomes.\nIn practice, we’ve helped hospitals implement AI tools that prioritize critical imaging, optimize scheduling, and identify high-risk patients for follow-up. These solutions reduce administrative burdens, enhance operational efficiency, and allow clinicians to focus on delivering quality care.\nAt OSP, we focus on integrating AI seamlessly into workflows, ensuring technology enhances human cap

In [32]:
tools = []

search_web_function = {
    "name" : "search_web",
    "description": "search the web using DuckDuckGo and return 3 results",
    "parameters": {
        "type": "object",
        "properties":{
            "query": {
                "type": "string",
                "description": "The search query information on the web"
            }
        },
        "required": ["query"]
    }
}

tools.append({"type":"function", "function": search_web_function} )

#-------------------------------------------

fetch_url_function = {
    "name" : "fetch_url",
    "description": "fetch url the main context from the web page",
    "parameters": {
        "type": "object",
        "properties":{
            "url": {
                "type": "string",
                "description": "fetch url extract text from the web"
            }
        },
        "required": ["url"]
    }
}

tools.append({"type":"function", "function": fetch_url_function} )


In [33]:
tools


[{'type': 'function',
  'function': {'name': 'search_web',
   'description': 'search the web using DuckDuckGo and return 3 results',
   'parameters': {'type': 'object',
    'properties': {'query': {'type': 'string',
      'description': 'The search query information on the web'}},
    'required': ['query']}}},
 {'type': 'function',
  'function': {'name': 'fetch_url',
   'description': 'fetch url the main context from the web page',
   'parameters': {'type': 'object',
    'properties': {'url': {'type': 'string',
      'description': 'fetch url extract text from the web'}},
    'required': ['url']}}}]

In [34]:
def handle_tool_call(tool_calls):
    tools_results = []
    # return what to add to our context (about tool call results, a dictionary)
    for tool_call in tool_calls:
        function_name = tool_call.function.name
        print(f" \U0001f527 Calling function {function_name}")
        args = json.loads(tool_call.function.arguments)

        #Route to function name
        if function_name == "search_web":

            result = search_web(args['query'])
            #print(f"Send notification: {args['message']}")
            content = f"Search Result: {result}"
        elif function_name == "fetch_url":
            result = fetch_url(args['url'])
            content = f"Fetch result: {result}"
        else:
            content = f"Unknown function {function_name}"

        tool_call_result  = {
            "role":"tool",
            "content":content,
            "tool_call_id":tool_call.id
            
        }

        tools_results.append(tool_call_result)

    return tools_results

## Step 4: System Prompt

In [35]:
RESEARCH_AGENT_PROMPT = """You are a research specialist. Your job is to research a given topic
and produce a comprehensive research brief.

You have access to two tools:
- search_web: Search the web for information
- fetch_url: Fetch and read the full content of a web page

Your typical process:
1. Search for the topic to find relevant sources
2. Reflect on the search results — which sources look most relevant and why?
3. Fetch the full content of the 2-3 best URLs
4. Reflect on what you have gathered. Do you have enough? Are there gaps?
5. If there are gaps, search again with a different query
6. When you have enough information from at least 3 different sources, synthesize into a research brief

When you are ready to deliver your final research brief, start your response with "DONE:" followed by the brief itself.

Your research brief MUST include:
- Key facts and statistics
- Main themes and arguments from the sources
- Notable data points
- Source URLs for attribution

Until you are ready, just keep working — search, fetch, think, reflect.
Do not rush. Take time to reflect between tool calls before deciding your next step.
Not every response needs a tool call — sometimes just thinking through what you have is the right move."""

## Step 5: Agentic Loop

In [42]:
from json import tool
from urllib import response


def run_research_agent(topic: str, max_iterations: int = 10) -> str:
    """ 
    Run a reseach a topic

    Args: 
        topic: The topic to research
        max_iternation: limit to prevent infinite loop
    Return: 
        The research brief

    """

    print("\n \U0001F50D Staring researching on topic: {topic}")
    print("=" * 60)

    messages = [
        {"role":"system", "content": RESEARCH_AGENT_PROMPT},
        {"role": "user", "content": f"Research the following topic and return a comprehensive research brief: \n\n {topic}"}
    ]

    iteration = 0
    while iteration < max_iterations:
        iteration += 1
        print(f"Iteratin {iteration}")

        response = client.chat.completions.create(
            model="gpt-4.1-mini",
            messages=messages,
            tools = tools,
            max_tokens=1000
        )

        message = response.choices[0].message
        messages.append(message)

        if message.tool_calls:
            tool_results = handle_tool_call(message.tool_calls)
            messages.extend(tool_results)
        else:
            content = message.content
            if content.startswith("DONE"):
                brief = content[len("DONE:"):].strip()
                print("\n \u2705 Research complete: \n{brief}")

                return brief
            else:

                print(f" \U0001F4AD Agent is thinking: ")
                pprint(content)

        if iteration == max_iterations - 1:
            print(" \u26A0 Stop researching in next iteration")
            messages.append({"role": "user", "content":"YOu have reached max iterations. Please deliver your brief with DONE"})
    
    #fall back return

    return "Research incomplete. Max iterations reached without finalizing brief."



In [ ]:
brief = run_research_agent("the future of AI in Ecommerce in 2030")
display(Markdown(brief))


 🔍 Staring researching on topic: {topic}
Iteratin 1
 🔧 Calling function search_web
("Got results: [{'title': 'AI in Healthcare 2030: Opportunities, Challenges, "
 "and Strategic Insights', 'href': "
 "'https://www.linkedin.com/pulse/ai-healthcare-2030-opportunities-challenges-strategic-riken-shah-pk86f', "
 "'body': 'Strategic Insights: Preparing For 2030. By 2030, AI will be an "
 'active partner, learning, adapting, and collaborating with clinicians. To '
 'leverage its potential: Invest in clinician AI literacy – Ensure staff '
 "understand AI’s capabilities, limitations, and ethical use.'}, {'title': "
 "'How will Artificial Intelligence Affect Jobs 2026-2030 | Nexford "
 "University', 'href': "
 "'https://www.nexford.edu/insights/how-will-ai-affect-jobs', 'body': 'AI will "
 'be taking some jobs, but it will be creating new ones! Here are the most '
 'likely jobs that artificial intelligence will affect from 2026-2030Forbes '
 'says that the future of AI brings endless possibilit

Research Brief on The Future of AI in Healthcare in 2030

Key Facts and Statistics:
- The global AI in healthcare market is projected to exceed USD 110 billion by 2030 with a compound annual growth rate (CAGR) of nearly 39% (DQ India).
- Healthcare data volume is expected to surpass 10 trillion gigabytes by late 2026, presenting a huge opportunity for AI to analyze complex and unstructured data (DQ India).
- Clinicians in the US spend nearly two hours on paperwork for every hour with patients, underlining the need for AI to reduce administrative burdens (DQ India).
- By 2030, AI is anticipated to be an active partner in healthcare, adapting and collaborating with clinicians, enhancing decision-making, workflow, and patient outcomes (LinkedIn article).
  
Main Themes and Arguments:
1. Transformational Role of AI:
   - AI will transform healthcare delivery by augmenting human expertise rather than replacing clinicians. It will improve clinical workflows, optimize scheduling, prioritize critical imaging, and identify high-risk patients, thereby enhancing operational efficiency (LinkedIn).
   - AI-driven tools like predictive analytics, smart diagnostic devices, and robotic-assisted surgeries will become commonplace to improve precision and individualized patient care (The Clinic Group Symposium).

2. Adoption Challenges:
   - Successful AI integration requires overcoming human readiness and integration barriers, including clinician AI literacy, ethical considerations, and seamless workflow embedding (LinkedIn).
   - Trust and data privacy remain pivotal issues, with companies like Anthropic and OpenAI focusing on secure, HIPAA-compliant healthcare AI platforms (DQ India).
   - Despite advances, AI error rates, especially in clinical decision-making (e.g., medication dosing), still pose risks that need stringent validation before AI moves from "drafting assistant" to "validated clinical partner" (DQ India).

3. AI in Administrative Relief and Economic Impact:
   - AI solutions can dramatically reduce administrative workload, such as billing, prior authorization, and medical coding, which currently contribute significantly to clinician burnout (DQ India).
   - Healthcare providers and insurers appear ready to invest in AI tools that promise cost reduction and improved patient outcomes, signaling a growing economic incentive for AI adoption (DQ India).

4. Ethical and Human-Centric Approach:
   - AI deployment strategies emphasize maintaining human oversight to ensure ethical use and patient-centered care.
   - Events like The Clinic Group Health Symposium 2025 highlight ongoing industry commitment to ethical AI, data privacy, and collaborative innovation between healthcare professionals and AI developers (The Clinic Group).

Notable Data Points:
- AI-powered voice assistants in healthcare are anticipated to be a key investment area by 2030, streamlining patient interactions and healthcare delivery (Violet Deer blog via search snippet).
- AI models like Claude for Healthcare and ChatGPT Health are leading innovation in creating specialized healthcare AI platforms with capabilities such as insurance navigation, claims processing, and medical record analysis, targeting both consumers and enterprises (DQ India).

Conclusion:
By 2030, AI will fundamentally reshape healthcare by enhancing diagnostic accuracy, operational efficiency, and patient-centered outcomes. The transition to widespread AI adoption hinges on strategic integration, clinician education, data privacy, and demonstrated clinical safety. The future landscape will see AI as a collaborative partner in medicine, enabling precision care and addressing critical workforce challenges like clinician burnout. Ethical considerations and maintaining human oversight remain paramount as AI technologies advance.

Sources:
- https://www.linkedin.com/pulse/ai-healthcare-2030-opportunities-challenges-strategic-riken-shah-pk86f
- https://www.dqindia.com/data-and-ai/from-chatbots-to-clinicians-the-logic-behind-the-ai-healthcare-surge-10991329
- https://theclinicgroup.org/2025/03/06/the-clinic-group-health-symposium-2025-the-future-of-medicine-unveiled-ais-role-in-healthcare-2030/